In [7]:
import requests
import pandas as pd
import json
from io import BytesIO
import unicodedata
import re

In [8]:
url = "https://static.data.gouv.fr/resources/finess-extraction-du-fichier-des-etablissements/20260512-091152/etalab-cs1100507-stock-20260512-0339.csv"

r = requests.get(url)
r.raise_for_status()

# D'abord, on regarde les vraies colonnes et le bon séparateur
df = pd.read_csv(BytesIO(r.content), sep=";", encoding="latin-1", low_memory=False, nrows=5)
print(df.columns.tolist())

['finess', 'etalab', '98', '2026-05-12']


In [9]:
def nettoyer_colonne(col):
    col = str(col).replace("\ufeff", "").strip().lower()
    col = "".join(
        c for c in unicodedata.normalize("NFKD", col)
        if not unicodedata.combining(c)
    )
    col = re.sub(r"[^a-z0-9]+", "_", col).strip("_")
    return col

# Lecture du fichier FINESS
df = pd.read_csv(
    url,
    sep=";",
    skiprows=1,          # important : ignore la ligne FINESS;ETALAB;...
    dtype=str,
    encoding="utf-8",
    low_memory=False
)

# Nettoyage des noms de colonnes
df.columns = [nettoyer_colonne(c) for c in df.columns]

print(df.columns.tolist())
print(df.shape)

['structureet', '010000024', '010780054', 'ch_de_fleyriat', 'centre_hospitalier_de_bourg_en_bresse_fleyriat', 'unnamed_5', 'unnamed_6', '900', 'rte', 'de_paris', 'unnamed_10', 'unnamed_11', '451', '01', 'ain', '01440_viriat', '0474454647', '0474454114', '355', 'centre_hospitalier_c_h', '1102', 'centres_hospitaliers', '26010004500012', '8610z', '03', 'ars_etablissements_publics_de_sante_dotation_globale', '1', 'etablissement_public_de_sante', '1979_02_13', '1979_02_13_1', '2020_02_04', 'unnamed_31']
(205877, 32)


In [10]:
url = "https://static.data.gouv.fr/resources/finess-extraction-du-fichier-des-etablissements/20260512-091152/etalab-cs1100507-stock-20260512-0339.csv"

colonnes = [
    "structureet",
    "nofinesset",
    "nofinessej",
    "rs",
    "rslongue",
    "complrs",
    "compldistrib",
    "numvoie",
    "typvoie",
    "voie",
    "compvoie",
    "lieuditbp",
    "commune",
    "departement",
    "libdepartement",
    "ligneacheminement",
    "telephone",
    "telecopie",
    "categetab",
    "libcategetab",
    "categagretab",
    "libcategagretab",
    "siret",
    "codeape",
    "codemft",
    "libmft",
    "codesph",
    "libsph",
    "dateouv",
    "dateautor",
    "datemaj",
    "numuai"
]

df = pd.read_csv(
    url,
    sep=";",
    skiprows=1,      # ignore la ligne FINESS;ETALAB;98;...
    header=None,     # le fichier n'a pas d'en-tête détaillé
    names=colonnes,
    dtype=str,
    encoding="utf-8",
    low_memory=False
)

print(df.shape)
df.head()

(205878, 32)


,structureet,nofinesset,nofinessej,rs,rslongue,complrs,compldistrib,numvoie,typvoie,voie,...,siret,codeape,codemft,libmft,codesph,libsph,dateouv,dateautor,datemaj,numuai
0,structureet,010000024,010780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,NaN,NaN,900,RTE,DE PARIS,...,26010004500012,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1979-02-13,1979-02-13,2020-02-04,NaN
1,structureet,010000032,010780062,CH BUGEY SUD,CENTRE HOSPITALIER BUGEY SUD,NaN,NaN,700,AV,DE NARVIK,...,26010003700068,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1901-01-01,1901-01-01,2021-07-07,NaN
2,structureet,010000065,010780096,CH DE TREVOUX - MONTPENSIER,CENTRE HOSPITALIER DE TREVOUX - MONTPENSIER,NaN,NaN,14,R,DE L'HOPITAL,...,26010028400017,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1901-01-01,1901-01-01,2018-01-12,NaN
3,structureet,010000081,010780112,CH DU PAYS DE GEX,CENTRE HOSPITALIER DU PAYS DE GEX,NaN,NaN,160,R,MARC PANISSOD,...,26010010200011,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1901-01-01,1901-01-01,2020-02-04,NaN
4,structureet,010000099,010780120,CH DE MEXIMIEUX,CENTRE HOSPITALIER DE MEXIMIEUX,NaN,NaN,13,AV,DU DOCTEUR BOYER,...,26010013600019,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1945-01-01,1945-01-01,2020-06-30,NaN


In [ ]:
# Code FINESS des EHPAD = 500
ehpad = df[
    df["categetab"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("500")
].copy()

print(f"Nombre d'EHPAD trouvés : {len(ehpad)}")

ehpad.head(100)

Nombre d'EHPAD trouvés : 7399


,structureet,nofinesset,nofinessej,rs,rslongue,complrs,compldistrib,numvoie,typvoie,voie,...,siret,codeape,codemft,libmft,codesph,libsph,dateouv,dateautor,datemaj,numuai
22,structureet,010002228,920030152,EHPAD L'AMBARROISE AMBERIEU,EHPAD L'AMBARROISE AMBERIEU,NaN,NaN,58,AV,PAUL PAINLEVE,...,49085207600010,8710A,47,"ARS/PCD, Tarif partiel, non habilité aide soci...",NaN,NaN,2006-12-04,2017-07-08,2026-05-07,NaN
106,structureet,010004059,010783009,EHPAD LE CLOS CHEVALIER ORNEX,EHPAD LE CLOS CHEVALIER ORNEX,NaN,NaN,7,R,PERE ADAM,...,77554456201205,8710A,41,"ARS/PCD, Tarif global, habilité aide sociale s...",NaN,NaN,2008-10-01,2022-12-18,2023-12-22,NaN
225,structureet,010006799,690802715,EHPAD LA ROSE DES VENTS,EHPAD LA ROSE DES VENTS,NaN,NaN,1289,R,EDOUARD HERRIOT,...,32735516000422,8710A,45,"ARS/PCD, Tarif partiel, habilité aide sociale ...",NaN,NaN,2011-03-01,2025-12-23,2026-01-06,NaN
297,structureet,010008571,010007987,EHPAD L'OREE DES SAPINS (CHPH),EHPAD L'OREE DES SAPINS (CHPH HAUTEVILLE),NaN,NaN,NaN,R,DES NARCISSES,...,26011019200028,NaN,40,"ARS/PCD, Tarif global, habilité aide sociale, ...",NaN,NaN,2009-10-19,2025-01-01,2025-01-21,NaN
334,structureet,010009223,010787109,EHPAD LES HELLEBORES GROISSIAT,EHPAD LES HELLEBORES GROISSIAT,MUTUALITE FRANCAISE AIN SSAM,NaN,1,PL,ST CYR,...,44429988700224,NaN,41,"ARS/PCD, Tarif global, habilité aide sociale s...",NaN,NaN,2016-09-01,2011-12-21,2025-11-06,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102480,structureet,970463063,970408589,EHPAD DE SAINT-JOSEPH,EHPAD DE SAINT-JOSEPH,NaN,NaN,NaN,R,RAPHAEL BABET,...,20003001300060,8710A,40,"ARS/PCD, Tarif global, habilité aide sociale, ...",NaN,NaN,1992-11-12,1987-12-17,2023-11-16,NaN
102706,structureet,970466256,970462396,EHPAD ASTERIA,EHPAD ASTERIA,NaN,NaN,5,ALL,BONNIER,...,32444658200052,NaN,41,"ARS/PCD, Tarif global, habilité aide sociale s...",NaN,NaN,1996-12-12,1991-12-19,2024-02-26,NaN
102714,structureet,970466652,970466645,E.H.P.A.D. LE MOUTARDIER,ET PR PERSONNES AGÉES DÉPENDANTES LE MOUTARDIER,NaN,NaN,15,CHE,MANES,...,41169020900027,NaN,41,"ARS/PCD, Tarif global, habilité aide sociale s...",NaN,NaN,1997-06-01,1992-12-28,2023-12-07,NaN
102717,structureet,970466728,750721334,E. H. P. A. D. CLOVIS HOARAU,ETS HEBERGEMENT PR PERSONNES AGEES DEPENDANTES...,NaN,NaN,42,R,DU BOIS DE NEFLES,...,77567227213499,8710A,41,"ARS/PCD, Tarif global, habilité aide sociale s...",NaN,NaN,1994-02-15,1991-03-25,2023-11-17,NaN
